# Ropedia Academy — A7 · Motion priors & generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/A7.ipynb)

> **Runs a full motion-diffusion reverse process (noise schedule + conditioned denoiser → a motion sample) and quantifies the foot-skating failure mode.**
>
> 跑完整的运动扩散反向过程（噪声调度 + 条件去噪器 → 一段运动样本），并量化脚滑这一失败模式。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/A7

In [ ]:
import torch, torch.nn as nn

T, D = 60, 24*6                                  # 60 frames, 24 joints x 6D rotation
betas = torch.linspace(1e-4, 0.02, 50)
abar  = torch.cumprod(1 - betas, 0)             # diffusion noise schedule

# A motion prior as a DIFFUSION model: learn to denoise; sample new motion (conditioned).
denoiser = nn.Sequential(nn.Linear(D + 1 + 16, 256), nn.SiLU(), nn.Linear(256, D))
def eps_pred(x, t, cond):
    tt = torch.full((x.shape[0], 1), t / len(betas))
    return denoiser(torch.cat([x, tt, cond], -1))

cond = torch.randn(1, 16)                        # text/music embedding ("a person waves")
x = torch.randn(1, D)                            # x_T: pure noise
for t in reversed(range(len(betas))):           # reverse process: denoise -> motion
    eps = eps_pred(x, t, cond)
    x = (x - (1 - abar[t]).sqrt() * eps) / abar[t].sqrt()
print("generated motion:", tuple(x.shape), "| conditioned on:", tuple(cond.shape))

# Plausibility is a GLOBAL physical property: foot-skate = foot moving while in contact.
foot_xy, contact = torch.randn(T, 2), (torch.rand(T) > 0.5)
skate = ((foot_xy[1:] - foot_xy[:-1]).norm(dim=-1) * contact[1:]).mean()
print("foot-skating penalty (lower is better):", round(skate.item(), 3))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/A7
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks